# Analise da riqueza especifica 

É feita a contabilização a partir da tabela de taxa e da tabela de ocorrências

nota: "Excluded species from Portugal"

### Importar tabela de ocorrências

In [1]:
import pandas as pd
import os

df_occ = pd.read_csv('../dados_raw/checklist_occurrences_29Abr2026.csv', low_memory=False)
df_prov_codes = pd.read_csv('../dados_raw/province_ids.csv')


df_occ_valid_taxa = df_occ[(df_occ['taxonomicStatus'].str.lower() == 'accepted') | (df_occ['taxonomicStatus'].str.lower() == 'synonym')].reset_index(drop=True)

df_occ_valid_taxa.shape


(43683, 62)

In [2]:
df_prov_codes

,provincia_process,PROVID,prov_code
0,Estremadura,7,ES
1,Minho,1,MI
2,Algarve,11,AL
3,Beira Alta,4,BA
4,Baixo Alentejo,10,BAl
5,Alto Alentejo,9,AA
6,Trás-os-Montes e Alto Douro,2,TM
7,Beira Litoral,5,BL
8,Douro Litoral,3,DL
9,Ribatejo,8,RI


In [3]:
# merge df_occ with df_prov_codes to get province codes
df_occ_valid_taxa = df_occ_valid_taxa.merge(df_prov_codes, left_on='provincia_process', right_on='provincia_process', how='left')

Verificar se há espécies aceites com mais do que um ID

In [4]:
# Group by acceptedNameUsageID and count unique acceptedNameUsage values
unique_counts = df_occ_valid_taxa.groupby('acceptedNameUsageID')['acceptedNameUsage'].nunique()

# Filter for IDs with more than one unique acceptedNameUsage
inconsistent_ids = unique_counts[unique_counts > 1].index.tolist()

# Display the list of inconsistent acceptedNameUsageID
print("acceptedNameUsageID with more than one acceptedNameUsage:")
print(inconsistent_ids)

acceptedNameUsageID with more than one acceptedNameUsage:
[]


In [5]:
df_sem_beira = df_occ_valid_taxa[(df_occ_valid_taxa['provincia_process'] != 'Beira')].reset_index(drop=True)

In [6]:
# create table with only provincia_process, habitat and acceptedNameUsageID columns, dropping rows with NA in any of these columns
df = df_sem_beira[['provincia_process', 'habitat', 'acceptedNameUsageID']].dropna()

In [7]:
# merge df with df_prov_codes to get province codes
df = df.merge(df_prov_codes, left_on='provincia_process', right_on='provincia_process', how='left') 
df

,provincia_process,habitat,acceptedNameUsageID,PROVID,prov_code
0,Estremadura,corticolous,9148e0a6c54475511f3ac26c3222045a,7,ES
1,Estremadura,corticolous,7620b0528ac0bada4c9f244c357283a3,7,ES
2,Estremadura,lichenicolous,d42c0868e238baac78bb55b1c3f400b6,7,ES
3,Estremadura,lichenicolous,d42c0868e238baac78bb55b1c3f400b6,7,ES
4,Trás-os-Montes e Alto Douro,lichenicolous,d42c0868e238baac78bb55b1c3f400b6,2,TM
...,...,...,...,...,...
34120,Beira Alta,muscicolous,71ed64cfe69b4dc8b57d08c1e92b9693,4,BA
34121,Beira Alta,muscicolous,71ed64cfe69b4dc8b57d08c1e92b9693,4,BA
34122,Beira Alta,muscicolous,71ed64cfe69b4dc8b57d08c1e92b9693,4,BA
34123,Beira Alta,muscicolous,71ed64cfe69b4dc8b57d08c1e92b9693,4,BA


## Numbers used in the paper
- number of occurrences
- number of occurrences with habitat data
- percentages per habitat type

In [8]:
print(f"Number of occurrences: {df_occ_valid_taxa.shape[0]}")

Number of occurrences: 43683


In [9]:
print(f"Number of occurrences with data in the habitat column: {df.shape[0]}")

Number of occurrences with data in the habitat column: 34125


In [10]:
# percentage of occurrences with data in the habitat column
perc_habitat = round(df.shape[0] / df_occ_valid_taxa.shape[0] * 100, 2)
print(f"Percentage of occurrences with data in the habitat column: {perc_habitat}%")  

Percentage of occurrences with data in the habitat column: 78.12%


In [11]:
# prompt: in df, the column 'habitat' contains multivalues representing a type of 
# habitat, separated by "|". I need a dataframe that contains, for each province, 
# the total number of species for each type of habitat

# count distinct species (acceptedNameUsageID) per provincia_process × habitat
cols = ['habitat', 'acceptedNameUsageID', 'PROVID','prov_code']
hab_df = df[cols].dropna(subset=['prov_code', 'habitat', 'acceptedNameUsageID', 'PROVID']).copy()
hab_df['hab'] = hab_df['habitat'].str.split(r'\s*\|\s*')
hab_df = hab_df.explode('hab')
hab_df['hab'] = hab_df['hab'].str.strip()

habitat_by_prov = (
    hab_df.groupby(['prov_code', 'PROVID', 'hab'], sort=True)['acceptedNameUsageID']
    .nunique()
    .reset_index(name='n_species')
)

province_species_total = (
  hab_df.groupby(['prov_code', 'PROVID'])['acceptedNameUsageID']
  .nunique()
  .reset_index(name='n_species_province')
)

habitat_by_prov = habitat_by_prov.merge(
  province_species_total,
  on=['prov_code', 'PROVID'],
  how='left'
)

habitat_by_prov['percent'] = (habitat_by_prov['n_species'] / habitat_by_prov['n_species_province'] * 100).round(2)

# optional: pivot to get provinces as rows and habitats as columns
habitat_by_prov_pivot = habitat_by_prov.pivot(index='prov_code', columns='hab', values='n_species').fillna(0).astype(int)

# optional: pivot to get provinces as rows and habitats as columns, with percentage values
habitat_by_prov_pivot_percent = habitat_by_prov.pivot(index='prov_code', columns='hab', values='percent').fillna(0).astype(float)


habitat_by_prov, habitat_by_prov_pivot

(   prov_code  PROVID            hab  n_species  n_species_province  percent
 0         AA       9     artificial         26                 394     6.60
 1         AA       9    corticolous        240                 394    60.91
 2         AA       9  lichenicolous         10                 394     2.54
 3         AA       9    lignicolous          2                 394     0.51
 4         AA       9    muscicolous         60                 394    15.23
 ..       ...     ...            ...        ...                 ...      ...
 74        TM       2  lichenicolous         83                 766    10.84
 75        TM       2    lignicolous         17                 766     2.22
 76        TM       2    muscicolous         86                 766    11.23
 77        TM       2     saxicolous        395                 766    51.57
 78        TM       2    terricolous        135                 766    17.62
 
 [79 rows x 6 columns],
 hab        artificial  corticolous  endolithic  f

### Percentages per habitat type

In [12]:
# based on hab_df count total number of rows per habitat, and its percentage of the total number of rows in hab_df
habitat_counts = hab_df['hab'].value_counts()
habitat_percent = (habitat_counts / habitat_counts.sum() * 100).round(2)
habitat_summary = pd.DataFrame({'count': habitat_counts, 'percent': habitat_percent})
habitat_summary

,count,percent
hab,,
corticolous,23184,61.39
saxicolous,8827,23.37
terricolous,2773,7.34
muscicolous,1565,4.14
lichenicolous,650,1.72
artificial,614,1.63
lignicolous,138,0.37
foliicolous,12,0.03
endolithic,1,0.00


In [13]:
# list unique values of habitat
habitat_by_prov['hab'].unique()

array(['artificial', 'corticolous', 'lichenicolous', 'lignicolous',
       'muscicolous', 'saxicolous', 'terricolous', 'foliicolous',
       'endolithic'], dtype=object)

In [14]:
# create dictionary for hab to hab_label mapping
hab_labels = {
    'artificial': 'artific',
    'corticolous': 'cortic',
    'endolithic': 'endol',
    'foliicolous': 'foliic',
    'lichenicolous': 'lichenic',
    'terricolous': 'terric',
    'saxicolous': 'saxic',
    'lignicolous': 'lignic',
    'muscicolous': 'muscic'
}

In [15]:
# in habitat_by_prov, replace hab values with hab_labels
# habitat_by_prov['hab'] = habitat_by_prov['hab'].map(hab_labels) 

In [16]:
habitat_by_prov

,prov_code,PROVID,hab,n_species,n_species_province,percent
0,AA,9,artificial,26,394,6.60
1,AA,9,corticolous,240,394,60.91
2,AA,9,lichenicolous,10,394,2.54
3,AA,9,lignicolous,2,394,0.51
4,AA,9,muscicolous,60,394,15.23
...,...,...,...,...,...,...
74,TM,2,lichenicolous,83,766,10.84
75,TM,2,lignicolous,17,766,2.22
76,TM,2,muscicolous,86,766,11.23
77,TM,2,saxicolous,395,766,51.57


In [17]:
habitat_national = (
    hab_df.groupby(['hab'], sort=True)['acceptedNameUsageID']
    .nunique()
    .reset_index(name='n_species')
)

habitat_national

,hab,n_species
0,artificial,198
1,corticolous,976
2,endolithic,1
3,foliicolous,7
4,lichenicolous,175
5,lignicolous,72
6,muscicolous,251
7,saxicolous,1079
8,terricolous,355


In [18]:
# add a column with the percentage of species in each habitat relative to the total number of species
total_species = habitat_national['n_species'].sum()
habitat_national['percent'] = (habitat_national['n_species'] / total_species * 100).round(2) 
habitat_national

,hab,n_species,percent
0,artificial,198,6.36
1,corticolous,976,31.34
2,endolithic,1,0.03
3,foliicolous,7,0.22
4,lichenicolous,175,5.62
5,lignicolous,72,2.31
6,muscicolous,251,8.06
7,saxicolous,1079,34.65
8,terricolous,355,11.40


In [19]:
# in habitat_national, replace hab values with hab_labels
# habitat_national['hab'] = habitat_national['hab'].map(hab_labels) 

In [20]:
# use plotly to create a radar plot representing the number of species for each habitat. 
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=habitat_national.sort_values('n_species', ascending=False)['n_species'],
    theta=habitat_national.sort_values('n_species', ascending=False)['hab'],
    fill='toself',
    name='Number of species'
))

fig.update_layout(
    polar=dict(
        angularaxis=dict(
            rotation=90,          # first value at top
            direction="clockwise" # clockwise order
        ),
        radialaxis=dict(
            visible=True,
            range=[0, habitat_national['n_species'].max() + 10]
        )
    ),
    showlegend=False,
    title='Number of species per habitat'
)   



In [21]:
# use plotly to create a radar plot representing the percentage of species for each habitat. 
fig_percent = go.Figure()
fig_percent.add_trace(go.Scatterpolar(
    r=habitat_national.sort_values('percent', ascending=False)['percent'],
    theta=habitat_national.sort_values('percent', ascending=False)['hab'],
    fill='toself',
    name='Percentage of species'
))
fig_percent.update_layout(
    polar=dict(
        angularaxis=dict(
            rotation=90,          # first value at top
            direction="clockwise" # clockwise order
        ),
        radialaxis=dict(
            visible=True,
            range=[0, habitat_national['percent'].max() + 10]
        )
    ),
    showlegend=False,
    title='Percentage of species per habitat'
)
fig.show()
fig_percent.show()



In [22]:
habitat_by_prov

,prov_code,PROVID,hab,n_species,n_species_province,percent
0,AA,9,artificial,26,394,6.60
1,AA,9,corticolous,240,394,60.91
2,AA,9,lichenicolous,10,394,2.54
3,AA,9,lignicolous,2,394,0.51
4,AA,9,muscicolous,60,394,15.23
...,...,...,...,...,...,...
74,TM,2,lichenicolous,83,766,10.84
75,TM,2,lignicolous,17,766,2.22
76,TM,2,muscicolous,86,766,11.23
77,TM,2,saxicolous,395,766,51.57


## Habitat max per province

In [23]:
# list which habitats are max in each province
habitat_by_prov['max_habitat'] = habitat_by_prov.groupby('prov_code')['n_species'].transform('max') == habitat_by_prov['n_species']
habitat_by_prov[habitat_by_prov['max_habitat']].groupby('prov_code')['hab'].apply(list).reset_index(name='max_habitats')

,prov_code,max_habitats
0,AA,[corticolous]
1,AL,[saxicolous]
2,BA,[saxicolous]
3,BAl,[corticolous]
4,BB,[corticolous]
5,BL,[corticolous]
6,DL,[saxicolous]
7,ES,[corticolous]
8,MI,[saxicolous]
9,RI,[corticolous]


In [24]:
# list of prov_code in habitat_by_prov sorted by PROVID
provs = habitat_by_prov.sort_values('PROVID')['prov_code'].unique()
provs

array(['MI', 'TM', 'DL', 'BA', 'BL', 'BB', 'ES', 'RI', 'AA', 'BAl', 'AL'],
      dtype=object)

In [25]:
# array of total number of species per province
total_species_prov = habitat_by_prov.sort_values('PROVID').groupby('prov_code')['n_species_province'].first()
total_species_prov = total_species_prov.sort_values(ascending=False)
total_species_prov

prov_code
ES     991
AL     833
TM     766
MI     638
BA     582
BL     520
AA     394
BAl    378
DL     367
BB     191
RI     168
Name: n_species_province, dtype: int64

In [26]:
type(total_species_prov)

pandas.core.series.Series

### Radar plot the number of species standardized

In [27]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- National reference radar (same for all subplots) ---
total_sorted = habitat_national.sort_values("n_species", ascending=False)
habitats_sorted = total_sorted["hab"].tolist()

short_theta = [hab_labels.get(h, h) for h in habitats_sorted]


# find angle for third axis
N = len(short_theta)
step = 360 / N
third_angle = 90 - 2 * step

# Create subplots
fig = make_subplots(
    rows=4,
    cols=3,
    specs=[[{"type": "polar"}] * 3 for _ in range(4)],
    subplot_titles=['Portugal'] + list(provs),
    vertical_spacing=0.08,
    horizontal_spacing=0.001
)

# ==================================================
# 1) NATIONAL RADAR AT TOP LEFT
# ==================================================
fig.add_trace(
    go.Scatterpolar(
        r=total_sorted["n_species"].values,
        theta=short_theta,
        fill="toself",
        name="Portugal",
        showlegend=False
    ),
    row=1, col=1
)

fig.update_layout({
    "polar": dict(
        bgcolor="white",
        angularaxis=dict(
            rotation=90, 
            direction="clockwise",
            tickfont=dict(size=18),
            showticklabels=True,
            gridcolor="darkgray",
            linecolor="darkgray"
        ),
        radialaxis=dict(
            angle=third_angle,
            visible=True,
            range=[0, total_sorted["n_species"].max() + 10],
            gridcolor="darkgray",
            linecolor="darkgray",
        )
    )
})


# ==================================================
# 2) PROVINCES START FROM SUBPLOT 2
# ==================================================
for idx, prov in enumerate(provs, start=1):
    subplot_pos = idx + 1   # shift by one because national uses first slot

    row = (subplot_pos - 1) // 3 + 1
    col = (subplot_pos - 1) % 3 + 1

    prov_data = habitat_by_prov_pivot.loc[prov].reindex(habitats_sorted)


    # province radar
    fig.add_trace(
        go.Scatterpolar(
            r=prov_data.values,
            theta=short_theta,
            fill="toself",
            name=prov,
            showlegend=False
        ),
        row=row,
        col=col
    )

    # national outline as reference
    fig.add_trace(
        go.Scatterpolar(
            r=total_sorted["n_species"].values,
            theta=short_theta,
            mode="lines",
            fill="toself",
            fillcolor="lightgray",
            line=dict(width=1, color="darkgray"),
            opacity=0.5,
            showlegend=False
        ),
        row=row,
        col=col
    )

    polar_name = f"polar{subplot_pos}"
    fig.update_layout({
        polar_name: dict(
            bgcolor="white",
            angularaxis=dict(
                rotation=90, 
                direction="clockwise",
                tickfont=dict(size=18),
                showticklabels=False,
                gridcolor="darkgray",
                linecolor="darkgray"
            ),
            radialaxis=dict(
                angle=third_angle,
                visible=True,
                range=[0, total_sorted["n_species"].max() + 10],
                gridcolor="darkgray",
                linecolor="darkgray",
            )
        )
    })

fig.update_layout(
    height=1600,
    width=1200,
    margin=dict(t=140),
    title_text="Number of species per substrate",
    title_x=0.5,
    title_y=.98,
    title_font_size=24,
    showlegend=False
)

# increase spacing between subplot title and radar
for annotation in fig.layout.annotations:
    annotation.y += 0.020
    annotation.font.size = 22


fig.show()

In [28]:
total_sorted

,hab,n_species,percent
7,saxicolous,1079,34.65
1,corticolous,976,31.34
8,terricolous,355,11.40
6,muscicolous,251,8.06
0,artificial,198,6.36
4,lichenicolous,175,5.62
5,lignicolous,72,2.31
3,foliicolous,7,0.22
2,endolithic,1,0.03


In [29]:
# save figure as high-res png
out = os.path.join("../images", "substrate_radar.png")
fig.write_image(out, format="png", width=1200, height=1600, scale=5)

out = os.path.join("../images", "substrate_radar.pdf")
fig.write_image(out, format="pdf", width=1200,height=1600, scale=1)


In [30]:
# calculate the total number of species in Portugal as the unique count of acceptedNameUsageID in the original df
tot_portugal = df_sem_beira['acceptedNameUsageID'].nunique()
tot_portugal

2010

## Radar number os species not standardized

In [31]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- National reference radar (same for all subplots) ---
total_sorted = habitat_national.sort_values("n_species", ascending=False)
habitats_sorted = total_sorted["hab"].tolist()

short_theta = [hab_labels.get(h, h) for h in habitats_sorted]

# find angle for third axis
N = len(short_theta)
step = 360 / N
third_angle = 90 - 2 * step

# Create subplots
fig = make_subplots(
    rows=4,
    cols=3,
    specs=[[{"type": "polar"}] * 3 for _ in range(4)],
    subplot_titles = [f"Portugal (n={tot_portugal})"] + [
        f"{prov} (n={total_species_prov.loc[prov]})"
        for prov in provs
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.04
)

# ==================================================
# 1) NATIONAL RADAR AT TOP LEFT
# ==================================================
fig.add_trace(
    go.Scatterpolar(
        r=total_sorted["n_species"].values,
        theta=short_theta,
        fill="toself",
        name="Portugal",
        showlegend=False
    ),
    row=1, col=1
)

fig.update_layout({
    "polar": dict(
        bgcolor="white",
        angularaxis=dict(
            rotation=90, 
            direction="clockwise",
            tickfont=dict(size=18),
            gridcolor="darkgray",
            linecolor="darkgray"
        ),
        radialaxis=dict(
            angle=third_angle,
            visible=True,
            gridcolor="darkgray",
            linecolor="darkgray",
        )
    )
})


# ==================================================
# 2) PROVINCES START FROM SUBPLOT 2
# ==================================================
for idx, prov in enumerate(provs, start=1):
    subplot_pos = idx + 1   # shift by one because national uses first slot

    row = (subplot_pos - 1) // 3 + 1
    col = (subplot_pos - 1) % 3 + 1

    prov_data = habitat_by_prov_pivot_percent.loc[prov].reindex(habitats_sorted)


    # province radar
    fig.add_trace(
        go.Scatterpolar(
            r=prov_data.values,
            theta=short_theta,
            fill="toself",
            fillcolor="plum",
            line_color="darkmagenta",
            opacity=0.5,
            name=prov,
            showlegend=False,
        ),
        row=row,
        col=col
    )

    polar_name = f"polar{subplot_pos}"
    fig.update_layout({
        polar_name: dict(
            bgcolor="white",             
            angularaxis=dict(
                rotation=90, 
                direction="clockwise",
                tickfont=dict(size=18),
                showticklabels=False,
                gridcolor="darkgray",
                linecolor="darkgray"
            ),
            radialaxis=dict(
                angle=third_angle,
                visible=True,
                # range=[0, total_sorted["percent"].max() + 10]
                gridcolor="darkgray",
                linecolor="darkgray"
            )
        )
    })

fig.update_layout(
    height=1600,
    width=1200,
    margin=dict(t=140),
    title_text="Number of species per substrate type",
    title_x=0.5,
    title_y=.98,
    title_font_size=24,
    showlegend=False
)

# increase spacing between subplot title and radar
for annotation in fig.layout.annotations:
    annotation.y += 0.020
    annotation.font.size = 22


fig.show()

In [32]:
# save figure as high-res png
out = os.path.join("../images", "substrate_radar_abs.png")
fig.write_image(out, format="png", width=1200, height=1600, scale=5)

out = os.path.join("../images", "substrate_radar_abs.pdf")
fig.write_image(out, format="pdf", width=1200,height=1600, scale=1)


In [33]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- National reference radar (same for all subplots) ---
total_sorted = habitat_national.sort_values("percent", ascending=False)
habitats_sorted = total_sorted["hab"].tolist()

short_theta = [hab_labels.get(h, h) for h in habitats_sorted]

# find angle for third axis
N = len(short_theta)
step = 360 / N
third_angle = 90 - 2 * step

# Create subplots
fig = make_subplots(
    rows=4,
    cols=3,
    specs=[[{"type": "polar"}] * 3 for _ in range(4)],
    subplot_titles = [f"Portugal (n={tot_portugal})"] + [
        f"{prov} (n={total_species_prov.loc[prov]})"
        for prov in provs
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.04
)

# ==================================================
# 1) NATIONAL RADAR AT TOP LEFT
# ==================================================
fig.add_trace(
    go.Scatterpolar(
        r=total_sorted["percent"].values,
        theta=short_theta,
        fill="toself",
        name="Portugal",
        showlegend=False
    ),
    row=1, col=1
)

fig.update_layout({
    "polar": dict(
        bgcolor="white",
        angularaxis=dict(
            rotation=90, 
            direction="clockwise",
            tickfont=dict(size=18),
            gridcolor="darkgray",
            linecolor="darkgray"
        ),
        radialaxis=dict(
            angle=third_angle,
            visible=True,
            gridcolor="darkgray",
            linecolor="darkgray",
        )
    )
})


# ==================================================
# 2) PROVINCES START FROM SUBPLOT 2
# ==================================================
for idx, prov in enumerate(provs, start=1):
    subplot_pos = idx + 1   # shift by one because national uses first slot

    row = (subplot_pos - 1) // 3 + 1
    col = (subplot_pos - 1) % 3 + 1

    prov_data = habitat_by_prov_pivot_percent.loc[prov].reindex(habitats_sorted)


    # province radar
    fig.add_trace(
        go.Scatterpolar(
            r=prov_data.values,
            theta=short_theta,
            fill="toself",
            fillcolor="firebrick",
            line_color="darkred",
            opacity=0.5,
            name=prov,
            showlegend=False,
        ),
        row=row,
        col=col
    )

    polar_name = f"polar{subplot_pos}"
    fig.update_layout({
        polar_name: dict(
            bgcolor="white",             
            angularaxis=dict(
                rotation=90, 
                direction="clockwise",
                tickfont=dict(size=18),
                showticklabels=False,
                gridcolor="darkgray",
                linecolor="darkgray"
            ),
            radialaxis=dict(
                angle=third_angle,
                visible=True,
                # range=[0, total_sorted["percent"].max() + 10]
                gridcolor="darkgray",
                linecolor="darkgray"
            )
        )
    })

fig.update_layout(
    height=1600,
    width=1200,
    margin=dict(t=140),
    title_text="Percentage of species per substrate",
    title_x=0.5,
    title_y=.98,
    title_font_size=24,
    showlegend=False
)

# increase spacing between subplot title and radar
for annotation in fig.layout.annotations:
    annotation.y += 0.020
    annotation.font.size = 22


fig.show()

In [34]:
# save figure as high-res png
out = os.path.join("../images", "substrate_radar_perc.png")
fig.write_image(out, format="png", width=1200, height=1600, scale=5)

out = os.path.join("../images", "substrate_radar_perc.pdf")
fig.write_image(out, format="pdf", width=1200,height=1600, scale=1)


In [35]:
# with df, create a barplot showing the number of occurrences 